In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Optional ARIMA
try:
    from statsmodels.tsa.arima.model import ARIMA
    ARIMA_AVAILABLE = True
except:
    ARIMA_AVAILABLE = False

print("Imports done")

Imports done


In [3]:
DATA_PATH = r"C:\Users\mansi.apet\Downloads\AI_SYSTEM (2)\AI_System(2)\AI_System_New_1\data\processed\subscription_with_patterns.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

df.head()

,CustomerID,TransactionID,Date,Description,Amount,Merchant,Status,Balance,is_recurring,frequency
0,1,15,2024-01-01,Gym Membership Subscription,300,Gym Membership,SUCCESS,9880,True,weekly
1,1,1,2024-01-05,Netflix Subscription,499,Netflix,SUCCESS,14567,True,monthly
2,1,2,2024-01-05,Netflix Subscription,499,Netflix,SUCCESS,14068,True,monthly
3,1,16,2024-01-08,Gym Membership Subscription,300,Gym Membership,SUCCESS,9580,True,weekly
4,1,41,2024-01-08,Electricity Bill Subscription,1200,Electricity Bill,SUCCESS,1180,False,irregular


In [4]:
df_rec = df[df["is_recurring"] == True].copy()

print("Rows:", len(df_rec))

Rows: 43539


In [5]:
def predict_debit(df):

    df_rec = df[df["is_recurring"] == True].copy()
    results = []

    for (cust, merchant), group in df_rec.groupby(["CustomerID", "Merchant"]):

        g = group.drop_duplicates("Date").sort_values("Date")

        if len(g) < 3:
            continue

        gaps = g["Date"].diff().dt.days.dropna()
        gaps = gaps[gaps > 0]

        avg_gap = gaps.mean()
        last_date = g["Date"].iloc[-1]
        mean_amount = g["Amount"].mean()
        freq = g.iloc[0]["frequency"]

        # Rule-based
        rule_next_date = last_date + pd.Timedelta(days=int(avg_gap))
        rule_next_amount = round(mean_amount, 2)

        # Linear Regression
        X = np.arange(len(g)).reshape(-1, 1)
        y = g["Date"].map(pd.Timestamp.toordinal)

        lr = LinearRegression().fit(X, y)
        next_ord = lr.predict([[len(g)]])[0]
        lr_next_date = pd.Timestamp.fromordinal(int(next_ord))

        lr_next_amount = rule_next_amount

        # Comparison
        date_diff = abs((rule_next_date - lr_next_date).days)
        amount_diff = abs(rule_next_amount - lr_next_amount)

        results.append({
            "CustomerID": cust,
            "Merchant": merchant,
            "last_date": last_date,
            "rule_next_date": rule_next_date,
            "lr_next_date": lr_next_date,
            "rule_next_amount": rule_next_amount,
            "lr_next_amount": lr_next_amount,
            "date_diff": date_diff,
            "amount_diff": amount_diff
        })

    return pd.DataFrame(results)

In [6]:
result_df = predict_debit(df)

result_df.head()

,CustomerID,Merchant,last_date,rule_next_date,lr_next_date,rule_next_amount,lr_next_amount,date_diff,amount_diff
0,1,Gym Membership,2024-06-24,2024-07-01,2024-07-01,300.0,300.0,0,0.0
1,1,Netflix,2024-06-05,2024-07-13,2024-07-17,499.0,499.0,4,0.0
2,1,Spotify,2024-06-10,2024-07-18,2024-07-22,199.0,199.0,4,0.0
3,2,Netflix,2024-06-05,2024-07-13,2024-07-17,499.0,499.0,4,0.0
4,2,Spotify,2024-06-10,2024-07-18,2024-07-22,199.0,199.0,4,0.0


In [7]:
def run_arima_demo(df):

    if not ARIMA_AVAILABLE:
        print("ARIMA not installed")
        return

    df_rec = df[df["is_recurring"] == True]

    # Pick ONE example
    g = df_rec[df_rec["Merchant"] == "Netflix"].sort_values("Date")

    if len(g) < 5:
        print("Not enough data for ARIMA")
        return

    model = ARIMA(g["Amount"].values, order=(1,1,1)).fit()
    pred = model.forecast(1)[0]

    print("ARIMA predicted amount:", round(pred, 2))

In [8]:
final_output = result_df.copy()

final_output.head()

,CustomerID,Merchant,last_date,rule_next_date,lr_next_date,rule_next_amount,lr_next_amount,date_diff,amount_diff
0,1,Gym Membership,2024-06-24,2024-07-01,2024-07-01,300.0,300.0,0,0.0
1,1,Netflix,2024-06-05,2024-07-13,2024-07-17,499.0,499.0,4,0.0
2,1,Spotify,2024-06-10,2024-07-18,2024-07-22,199.0,199.0,4,0.0
3,2,Netflix,2024-06-05,2024-07-13,2024-07-17,499.0,499.0,4,0.0
4,2,Spotify,2024-06-10,2024-07-18,2024-07-22,199.0,199.0,4,0.0


In [9]:
OUT_PATH = r"C:\Users\Mansi\Downloads\OPUS PYTHON\AI_System_New_1\AI_System_New_1\data\output\debit_predictions.csv"

final_output.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)

OSError: Cannot save file into a non-existent directory: 'C:\Users\Mansi\Downloads\OPUS PYTHON\AI_System_New_1\AI_System_New_1\data\output'